# 06a - Select and Freeze Winner

CPU only. Select from completed candidates using recorded development and strategy-validation evidence, freeze the choice, then measure its held-out general and strategy performance.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Project and Dependencies

Keep Colab's existing CUDA PyTorch. Restart only if pip explicitly requires it.

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/Education/Deep Reinforcement Learning/Chess")
if not (PROJECT_ROOT / "chess_rl").is_dir():
    raise FileNotFoundError(f"Project files are missing from {PROJECT_ROOT}. See README.md.")
sys.path.insert(0, str(PROJECT_ROOT))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-r", str(PROJECT_ROOT / "requirements_colab.txt")])

## Run Configuration

Edit configs/default.yaml once for the workflow. Use a new run_id for a changed experiment.

In [ ]:
from chess_rl.config import load_config, prepare_directories
from chess_rl.reproducibility import metadata, read_json, atomic_json, sha256

prepare_directories(PROJECT_ROOT)
cfg = load_config(PROJECT_ROOT, "strategy.yaml")
print("Run:", cfg["run_id"])
print("Runtime:", metadata())

## Selection Criteria

Reject candidates with runtime/model errors or unusable development results. Apply configured minimum general score and optional strategy-pass requirement. Rank by mean opponent score, then candidate id for deterministic ties; strategy scores are diagnostic by default. Only the leading challenger faces the fixed classical incumbent on the separate 256-game confirmation suite. Replacement requires at least 55% score, paired-bootstrap lower bound above 50%, and no runtime failures. Failure retains the incumbent; never try another challenger on the same suite. The frozen winner cannot be replaced using its test results.

In [ ]:
print(cfg["research_workflow"]["selection"])

## Freeze and Measure Held-Out Performance

Lock source hashes, configuration, checkpoint hash, opening hashes, evaluation summaries and ranking before held-out games. The chosen model alone runs test suites. The final selection works without a self-play checkpoint, including when a classical candidate wins.

In [ ]:
from chess_rl.research_workflow import freeze_winner
selection_path = freeze_winner(PROJECT_ROOT, cfg)
selection = read_json(selection_path)
print("Selected:", selection["selected_id"], selection["kind"])
print("Frozen manifest:", selection_path)
print("Next: final export notebook 06b.")